### Step 1 — Load the January data
Opens `training_2025-01-01_2025-02-01.parquet` and loads it into a table called `df`, using the pandas library.

In [6]:
import pandas as pd

df = pd.read_parquet('/content/training_2025-01-01_2025-02-01.parquet')

### Step 2 — First look at the data
Checks the table's size (rows, columns) and previews the first 5 rows, just to see what the raw data actually looks like.

In [7]:
print(df.shape)
df.head()

(307257, 30)


,MVT_ID_mvt,FLIGHT_ID_mvt,FLIGHT_mvt,FLIGHT_RULE_mvt,ADEP_mvt,ADES_mvt,PHASE_mvt,MVT_TIME_UTC_mvt,BLOCK_TIME_UTC_mvt,SCHED_TIME_UTC_mvt,...,IOBT_flt,FLIGHT_RULE_flt,FLIGHT_TYPE_flt,AIRCRAFT_TYPE_flt,WK_TBL_CAT_flt,AIRCRAFT_OPERATOR_flt,EOBT_1_flt,ARVT_1_flt,AOBT_3_flt,ARVT_3_flt
0,182234017.0,280602939.0,LH982,I,EDDF,EIDW,DEP,2025-01-07 21:10:06+00:00,2025-01-07 21:00:11+00:00,2025-01-07 21:00:00+00:00,...,2025-01-07 21:00:00+00:00,I,S,A320,M,11dca3faf5500fdddbb17189b5d18bcdc763244fcc850c...,2025-01-07 21:00:00+00:00,2025-01-07 23:11:12+00:00,2025-01-07 20:57:00+00:00,2025-01-07 22:58:31+00:00
1,182249437.0,280601336.0,LO383,I,EPWA,EDDF,ARR,2025-01-07 21:10:32+00:00,2025-01-07 21:18:57+00:00,2025-01-07 21:10:00+00:00,...,2025-01-07 19:22:00+00:00,I,S,E170,M,1e218b1a730a8fe295a6547b48d215268be4ae3030d571...,2025-01-07 19:22:00+00:00,2025-01-07 21:15:43+00:00,2025-01-07 19:16:00+00:00,2025-01-07 21:10:28+00:00
2,182234011.0,280602803.0,LH796,I,EDDF,VHHH,DEP,2025-01-07 21:11:23+00:00,2025-01-07 20:53:18+00:00,2025-01-07 20:50:00+00:00,...,2025-01-07 20:50:00+00:00,I,S,A346,H,11dca3faf5500fdddbb17189b5d18bcdc763244fcc850c...,2025-01-07 20:50:00+00:00,2025-01-08 08:21:09+00:00,2025-01-07 20:53:00+00:00,2025-01-08 08:22:32+00:00
3,182234021.0,280602920.0,LH958,I,EDDF,EGBB,DEP,2025-01-07 21:12:29+00:00,2025-01-07 21:01:01+00:00,2025-01-07 21:00:00+00:00,...,2025-01-07 21:00:00+00:00,I,S,A320,M,11dca3faf5500fdddbb17189b5d18bcdc763244fcc850c...,2025-01-07 21:00:00+00:00,2025-01-07 22:33:40+00:00,2025-01-07 21:01:00+00:00,2025-01-07 22:36:05+00:00
4,182249434.0,280602048.0,LH121,I,EDDM,EDDF,ARR,2025-01-07 21:13:01+00:00,2025-01-07 21:18:14+00:00,2025-01-07 21:05:00+00:00,...,2025-01-07 20:00:00+00:00,I,S,A320,M,11dca3faf5500fdddbb17189b5d18bcdc763244fcc850c...,2025-01-07 20:00:00+00:00,2025-01-07 21:01:21+00:00,2025-01-07 20:16:00+00:00,2025-01-07 21:13:45+00:00


### Step 3 — Check for missing data
For every column, calculates what percentage of its values are empty. Shows the worst offenders first.

In [8]:
df.isna().mean().sort_values(ascending=False) * 100

,0
ARVT_1_flt,0.826995
EOBT_1_flt,0.826995
AIRCRAFT_OPERATOR_flt,0.826995
WK_TBL_CAT_flt,0.826995
AIRCRAFT_TYPE_flt,0.826995
FLIGHT_TYPE_flt,0.826995
FLIGHT_RULE_flt,0.826995
IOBT_flt,0.826995
MARKET_SEGMENT_flt,0.826995
ADES_FILED_flt,0.826995


### Step 4 — Flag rows with no flight-plan data
Creates a new column, `has_flight_plan_data`, marking `False` for the ~0.8% of rows where the flight-plan (`_flt`) columns are all missing. Nothing is deleted — just flagged.

In [9]:
df['has_flight_plan_data'] = df['ADEP_flt'].notna()

### Step 5 — Confirm the flag worked
Counts how many rows are `True`/`False` in the new flag column, to sanity-check Step 4.

In [10]:
df['has_flight_plan_data'].value_counts()

,count
has_flight_plan_data,
True,304716
False,2541


### Step 6 — Find impossible taxi times
Counts how many rows have a taxi time of zero or negative seconds — physically impossible, so these are data errors.

In [11]:
(df['TAXITIME_SEC_mvt'] <= 0).sum()

np.int64(79)

### Step 7 — Flag the impossible taxi times
Creates `taxi_time_valid`, marking `False` for any row where taxi time is ≤ 0. Again, flagged rather than deleted.

In [12]:
df['taxi_time_valid'] = df['TAXITIME_SEC_mvt'] > 0

### Step 8 — Check for duplicate flight records
Counts how many times the same `MVT_ID_mvt` (unique flight movement ID) appears more than once. Result: 0 duplicates in January.

In [13]:
df['MVT_ID_mvt'].duplicated().sum()

np.int64(0)

### Step 9 — Count departures vs arrivals
Counts how many rows are `DEP` (departures) vs `ARR` (arrivals) — turned out to be almost a 50/50 split.

In [14]:
df['PHASE_mvt'].value_counts()

,count
PHASE_mvt,
DEP,153706
ARR,153551


### Step 10 — Compare taxi times by phase
Summarizes taxi time separately for DEP and ARR rows (average, min, max, etc.) to understand whether they behave differently. This is what led to the decision to keep both phases rather than dropping ARR rows.

In [15]:
df.groupby('PHASE_mvt')['TAXITIME_SEC_mvt'].describe()

,count,mean,std,min,25%,50%,75%,max
PHASE_mvt,,,,,,,,
ARR,153551.0,544.834824,320.453451,-1283.0,340.0,486.0,671.0,33601.0
DEP,153706.0,995.292526,604.833035,-12.0,714.0,906.0,1175.0,87177.0


### Step 11 — Look at unusually long taxi times
Pulls up every row with a taxi time over 2 hours (7200 seconds), to eyeball how extreme the long ones actually get.

In [16]:
df[df['TAXITIME_SEC_mvt'] > 7200][['FLIGHT_mvt','PHASE_mvt','TAXITIME_SEC_mvt']]

,FLIGHT_mvt,PHASE_mvt,TAXITIME_SEC_mvt
40719,FCK1EL,ARR,12245
62397,SHT2J,DEP,10210
63472,SHT12Y,DEP,9611
64710,SHT18Z,DEP,10615
73527,BAW12P,DEP,8232
77469,BAW48HY,DEP,7267
78748,BAW48HY,DEP,9468
79458,BAW430A,DEP,8275
89845,BBC201,ARR,7207
102278,NCG01A,DEP,11032


### Step 12 — Count taxi times at stricter cutoffs
Checks how many rows are over 4 hours and over 12 hours of taxi time, to help pick a sensible line between "rare but real" and "definitely broken data."

In [17]:
print('over 4 hours (14400s):', (df['TAXITIME_SEC_mvt'] > 14400).sum())
print('over 12 hours (43200s):', (df['TAXITIME_SEC_mvt'] > 43200).sum())

over 4 hours (14400s): 9
over 12 hours (43200s): 4


### Step 13 — Flag the true outliers
Creates `taxi_time_outlier`, marking `True` for rows where taxi time is over 12 hours (43,200 seconds) — not physically realistic, almost certainly a data error.

In [18]:
df['taxi_time_outlier'] = df['TAXITIME_SEC_mvt'] > 43200

### Step 14 — Confirm the outlier flag
Counts how many rows got flagged as outliers (4, in January).

In [19]:
df['taxi_time_outlier'].sum()

np.int64(4)

### Step 15 — See all 3 flags together
Shows a summary count of all three quality flags (`has_flight_plan_data`, `taxi_time_valid`, `taxi_time_outlier`) side by side — the full picture of what's been cleaned so far, still just on January.

In [20]:
df[['has_flight_plan_data','taxi_time_valid','taxi_time_outlier']].sum()

,0
has_flight_plan_data,304716
taxi_time_valid,307178
taxi_time_outlier,4


### Step 16 — Turn the recipe into a reusable function
Packages everything done so far (load file → add 3 flags) into one function, `clean_month()`, so it can be reused on any monthly file without retyping the steps.

In [21]:
def clean_month(filepath):
    df = pd.read_parquet(filepath)
    df['has_flight_plan_data'] = df['ADEP_flt'].notna()
    df['taxi_time_valid'] = df['TAXITIME_SEC_mvt'] > 0
    df['taxi_time_outlier'] = df['TAXITIME_SEC_mvt'] > 43200
    return df

### Step 17 — Test the function on January
Runs `clean_month()` on the January file to confirm it produces the exact same result as doing it manually, cell by cell.

In [22]:
df_jan = clean_month('/content/training_2025-01-01_2025-02-01.parquet')
df_jan.shape

(307257, 33)

### Step 18 — Find all 12 monthly files
Uses `glob` to automatically find every uploaded file starting with `training_`, sorted in date order — so all 12 months can be looped through automatically.

In [23]:
import glob

files = sorted(glob.glob('/content/training_*.parquet'))
files

['/content/training_2025-01-01_2025-02-01.parquet',
 '/content/training_2025-02-01_2025-03-01.parquet',
 '/content/training_2025-03-01_2025-04-01.parquet',
 '/content/training_2025-04-01_2025-05-01.parquet',
 '/content/training_2025-05-01_2025-06-01.parquet',
 '/content/training_2025-06-01_2025-07-01.parquet',
 '/content/training_2025-07-01_2025-08-01.parquet',
 '/content/training_2025-08-01_2025-09-01.parquet',
 '/content/training_2025-09-01_2025-10-01.parquet',
 '/content/training_2025-10-01_2025-11-01.parquet',
 '/content/training_2025-11-01_2025-12-01.parquet',
 '/content/training_2025-12-01_2026-01-01.parquet']

### Step 19 — Clean all 12 months
Loops through all 12 files, runs `clean_month()` on each one, and collects the cleaned tables in a list (`all_months`), printing each month's size along the way.

In [24]:
all_months = []

for f in files:
    cleaned = clean_month(f)
    all_months.append(cleaned)
    print(f, cleaned.shape)

/content/training_2025-01-01_2025-02-01.parquet (307257, 33)
/content/training_2025-02-01_2025-03-01.parquet (287411, 33)
/content/training_2025-03-01_2025-04-01.parquet (328622, 33)
/content/training_2025-04-01_2025-05-01.parquet (350288, 33)
/content/training_2025-05-01_2025-06-01.parquet (370105, 33)
/content/training_2025-06-01_2025-07-01.parquet (365986, 33)
/content/training_2025-07-01_2025-08-01.parquet (381161, 33)
/content/training_2025-08-01_2025-09-01.parquet (382047, 33)
/content/training_2025-09-01_2025-10-01.parquet (367679, 33)
/content/training_2025-10-01_2025-11-01.parquet (371163, 33)
/content/training_2025-11-01_2025-12-01.parquet (324530, 33)
/content/training_2025-12-01_2026-01-01.parquet (331548, 33)


### Step 20 — Combine into one dataset
Stacks all 12 cleaned monthly tables into a single table, `full_data`, covering the whole year (4,167,797 rows total).

In [25]:
full_data = pd.concat(all_months, ignore_index=True)
full_data.shape

(4167797, 33)

### Step 21 — Save the cleaned dataset
Saves `full_data` to disk as `cleaned_full_data.parquet`, so the cleaning work doesn't need to be repeated.

In [26]:
full_data.to_parquet('/content/cleaned_full_data.parquet', index=False)

### Step 22 — Final summary across the full year
Prints the full-year totals for all 3 flags, plus the DEP/ARR split — the real numbers to report on the cleaning work (not just January's).

In [27]:
print('Total rows:', full_data.shape[0])
print()
print('Missing flight-plan data:', (~full_data['has_flight_plan_data']).sum())
print('Invalid (<=0) taxi times:', (~full_data['taxi_time_valid']).sum())
print('Outlier (>12hr) taxi times:', full_data['taxi_time_outlier'].sum())
print()
print(full_data['PHASE_mvt'].value_counts())

Total rows: 4167797

Missing flight-plan data: 37919
Invalid (<=0) taxi times: 625
Outlier (>12hr) taxi times: 39

PHASE_mvt
DEP    2085047
ARR    2082750
Name: count, dtype: int64


### Step 23 — Check data types
Lists what type of data each column holds (text, number, date/time, True/False), to confirm nothing is stored as the wrong type (e.g. dates stored as plain text).

In [28]:
full_data.dtypes

,0
MVT_ID_mvt,float64
FLIGHT_ID_mvt,float64
FLIGHT_mvt,object
FLIGHT_RULE_mvt,object
ADEP_mvt,object
ADES_mvt,object
PHASE_mvt,object
MVT_TIME_UTC_mvt,"datetime64[us, UTC]"
BLOCK_TIME_UTC_mvt,"datetime64[us, UTC]"
SCHED_TIME_UTC_mvt,"datetime64[us, UTC]"


### Step 24 — Duplicate check on the full dataset
Re-runs the duplicate-ID check from Step 8, but this time against all 4.1 million rows across the full year, not just January. Result: 0 duplicates.

In [29]:
full_data['MVT_ID_mvt'].duplicated().sum()

np.int64(0)

### Step 25 — Run the packaged script
Runs the final, reusable `clean_taxi_data.py` script end-to-end in one command — doing everything above (load, clean, combine, check, save) automatically, reproducing the same results.

In [30]:
!python clean_taxi_data.py

/content/training_2025-01-01_2025-02-01.parquet: (307257, 33)
/content/training_2025-02-01_2025-03-01.parquet: (287411, 33)
/content/training_2025-03-01_2025-04-01.parquet: (328622, 33)
/content/training_2025-04-01_2025-05-01.parquet: (350288, 33)
/content/training_2025-05-01_2025-06-01.parquet: (370105, 33)
/content/training_2025-06-01_2025-07-01.parquet: (365986, 33)
/content/training_2025-07-01_2025-08-01.parquet: (381161, 33)
/content/training_2025-08-01_2025-09-01.parquet: (382047, 33)
/content/training_2025-09-01_2025-10-01.parquet: (367679, 33)
/content/training_2025-10-01_2025-11-01.parquet: (371163, 33)
/content/training_2025-11-01_2025-12-01.parquet: (324530, 33)
/content/training_2025-12-01_2026-01-01.parquet: (331548, 33)

--- Sanity checks ---
Total rows: 4167797
Duplicate MVT_ID_mvt: 0
Missing flight-plan data: 37919
Invalid (<=0) taxi times: 625
Outlier (>12hr) taxi times: 39

Rows by phase:
PHASE_mvt
DEP    2085047
ARR    2082750
Name: count, dtype: int64

Saved cleaned